#36615 - Final Project Part 1 - Data Analysis and Visualizations
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

#### Description:
This notebook performs exploratory analysis on the cleaned data.

In [0]:
# import necessary libraries for aggregation and plotting
import pyspark.sql.functions as F
import pyspark.sql.types as T
import matplotlib.pyplot as plt
import pandas as pd

In [0]:
# load the cleaned data from the metastore
delays = spark.table("lsd_2026.default.polyhymnia_cleaned_airline")
print(f"Loaded {delays.count()} rows")

#### Task 1 - Number of flights per month across all years

In [0]:
# extract year and month from FL_DATE, count flights
flights_per_month = delays.groupBy(
    F.year("FL_DATE").alias("year"),
    F.month("FL_DATE").alias("month")
).agg(
    F.count("*").alias("num_flights")
).orderBy("year", "month")


In [0]:
# show the first few oldest and latest records
flights_per_month_ends = flights_per_month.limit(5) \
    .union(flights_per_month.orderBy(F.desc("year"), F.desc("month")) \
    .limit(5).orderBy(F.asc("year"), F.asc("month")))
flights_per_month_ends.show()

This verifies we've loaded all the data correctly and shows the temporal coverage of the dataset as spanning 10 years beginning in January 2019 and ending in December 2018. A time series plot of the number of flights per month in this time span is given below.

In [0]:
# convert to Pandas for visualization with matplotlib
flights_per_month_pd = flights_per_month.toPandas()

# create year-month column for easier plotting
flights_per_month_pd['date'] = pd.to_datetime(
    flights_per_month_pd['year'].astype(str) + '-' + 
    flights_per_month_pd['month'].astype(str)
)

# plot the time series
plt.figure(figsize=(14, 6))
plt.plot(flights_per_month_pd['date'], flights_per_month_pd['num_flights'], 
         linewidth=2, color='#1f77b4')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Flights', fontsize=12)
plt.title('Number of Flights per Month', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# summary statistics
print(f"\nDate range: {flights_per_month_pd['date'].min()} to {flights_per_month_pd['date'].max()}")
print(f"Total months: {len(flights_per_month_pd)}")
print(f"Average flights per month: {flights_per_month_pd['num_flights'].mean():.0f}")

The plot shows flight data spans from 2009 to end of 2018, and reveals two spikes in the fall of 2009 and the summer of 2011. On average, there were 521,669 flights per month in this time range.

#### Task 2 - Percentage of flights delayed per week

In [0]:
# calculate week of year and year for grouping
# a flight is delayed if arrival delay > 0 OR it was cancelled
flights_by_week = delays.groupBy(
    F.year("FL_DATE").alias("year"),
    F.weekofyear("FL_DATE").alias("week")
).agg(
    F.count("*").alias("total_flights"),    # find total flights for that week
    F.sum(  # count up the number of delayed flights
        F.when((F.col("ARR_DELAY") > 0) | (F.col("CANCELLED") == 1), 1)
        .otherwise(0)
    ).alias("delayed_flights")
).orderBy("year", "week")

# calculate percentage delayed
flights_by_week = flights_by_week.withColumn(
    "pct_delayed",
    (F.col("delayed_flights") / F.col("total_flights")) * 100
)

# convert to Pandas for visualization
flights_by_week_pd = flights_by_week.toPandas()

# create date column for plotting (using Monday to represent a week)
flights_by_week_pd['date'] = pd.to_datetime(
    flights_by_week_pd['year'].astype(str) + '-W' + 
    flights_by_week_pd['week'].astype(str) + '-1',
    format='%Y-W%W-%w'
)

# make the plot of percentage delayed over time
plt.figure(figsize=(14, 6))
plt.plot(flights_by_week_pd['date'], flights_by_week_pd['pct_delayed'], 
         linewidth=1, color='#d62728', alpha=0.7)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Percentage Delayed (%)', fontsize=12)
plt.title('Percentage of Flights Delayed per Week', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# summary stats
print(f"\nOverall delay rate: {flights_by_week_pd['pct_delayed'].mean():.2f}%")
print(f"Highest weekly delay rate: {flights_by_week_pd['pct_delayed'].max():.2f}%")
print(f"Lowest weekly delay rate: {flights_by_week_pd['pct_delayed'].min():.2f}%")


Across the 10 years of data, the average weekly percentage of delayed flights was 38.17%. Percentage of weekly delays typically seem to increase around holiday season.

#### Task 3 - Number of delayed flights per week, by delay type

In [0]:
# group by week and sum each delay type
# only count a flight if it actually had that type of delay
delays_by_type = delays.groupBy(
    F.year("FL_DATE").alias("year"),
    F.weekofyear("FL_DATE").alias("week")
).agg(
    F.sum(F.when(F.col("CARRIER_DELAY") > 0, 1).otherwise(0)).alias("carrier_delays"),
    F.sum(F.when(F.col("WEATHER_DELAY") > 0, 1).otherwise(0)).alias("weather_delays"),
    F.sum(F.when(F.col("NAS_DELAY") > 0, 1).otherwise(0)).alias("nas_delays"),
    F.sum(F.when(F.col("SECURITY_DELAY") > 0, 1).otherwise(0)).alias("security_delays"),
    F.sum(F.when(F.col("LATE_AIRCRAFT_DELAY") > 0, 1).otherwise(0)).alias("late_aircraft_delays")
).orderBy("year", "week")

# convert to Pandas for visualization
delays_by_type_pd = delays_by_type.toPandas()

# create date column for plotting
delays_by_type_pd['date'] = pd.to_datetime(
    delays_by_type_pd['year'].astype(str) + '-W' + 
    delays_by_type_pd['week'].astype(str) + '-1',
    format='%Y-W%W-%w'
)

# plot all delay types on one chart
plt.figure(figsize=(14, 7))
plt.plot(delays_by_type_pd['date'], delays_by_type_pd['carrier_delays'], 
         label='Carrier', linewidth=1, alpha=0.8)
plt.plot(delays_by_type_pd['date'], delays_by_type_pd['weather_delays'], 
         label='Weather', linewidth=1, alpha=0.8)
plt.plot(delays_by_type_pd['date'], delays_by_type_pd['nas_delays'], 
         label='NAS', linewidth=1, alpha=0.8)
plt.plot(delays_by_type_pd['date'], delays_by_type_pd['security_delays'], 
         label='Security', linewidth=1, alpha=0.8)
plt.plot(delays_by_type_pd['date'], delays_by_type_pd['late_aircraft_delays'], 
         label='Late Aircraft', linewidth=1, alpha=0.8)

plt.xlabel('Date', fontsize=12)
plt.ylabel('Number of Delayed Flights', fontsize=12)
plt.title('Number of Delayed Flights per Week by Delay Type', fontsize=14, fontweight='bold')
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# some summary statistics of averages by type
print("\nAverage weekly delays by type:")
print(f"  Carrier: {delays_by_type_pd['carrier_delays'].mean():.0f}")
print(f"  Weather: {delays_by_type_pd['weather_delays'].mean():.0f}")
print(f"  NAS: {delays_by_type_pd['nas_delays'].mean():.0f}")
print(f"  Security: {delays_by_type_pd['security_delays'].mean():.0f}")
print(f"  Late Aircraft: {delays_by_type_pd['late_aircraft_delays'].mean():.0f}")


Security seems to be the least common reason for delay (almost a flat line at 0), followed by weather which experiences peaks and troughs throughout the year. The other three reasons (NAS, Late Aircraft, and Carrier) seem to be the primary drivers of delays and are closely linked.

#### Task 4 - Air carrier performance summary table

In [0]:
# for each carrier, find total flights, cancellations, % delayed, and avg delay
# sort by total flights (largest carriers first)
carrier_performance = delays.groupBy("OP_CARRIER").agg(
    F.count("*").alias("total_flights"),
    F.sum(F.col("CANCELLED")).alias("cancelled_flights"),
    F.sum(
        F.when((F.col("ARR_DELAY") > 0) | (F.col("CANCELLED") == 1), 1)
        .otherwise(0)
    ).alias("delayed_flights"),
    F.avg(
        F.when(F.col("ARR_DELAY") > 0, F.col("ARR_DELAY"))  # F.avg() ignores null values
    ).alias("avg_delay_minutes")
).orderBy(F.desc("total_flights"))

# rename the carrier column for intuitive display
carrier_performance = carrier_performance.withColumnRenamed("OP_CARRIER", "air_carrier")

# calculate percentage delayed
carrier_performance = carrier_performance.withColumn(
    "pct_delayed",
    (F.col("delayed_flights") / F.col("total_flights")) * 100
)

# select and reorder columns for final display
carrier_performance = carrier_performance.select("air_carrier", "total_flights", "cancelled_flights", "pct_delayed", "avg_delay_minutes")

# show the resulting summary table of all 23 air carriers
carrier_performance.show(carrier_performance.count())


#### Task 5 - Top 50 airports by percentage of flights delayed

In [0]:
# derive the total date range in the dataset (should be 3650 days for ten years)
date_range = delays.agg(
    F.min("FL_DATE").alias("min_date"),
    F.max("FL_DATE").alias("max_date")
).collect()[0] # collect turns df into a list of rows

min_date = date_range['min_date']
max_date = date_range['max_date']
total_days = (max_date - min_date).days + 1 # inclusive

print(f"Date range: {min_date} to {max_date}")
print(f"Total days: {total_days}")

# calculate delay statistics for each origin airport
airport_performance = delays.groupBy("ORIGIN").agg(
    F.count("*").alias("total_flights"),
    F.sum(
        F.when((F.col("ARR_DELAY") > 0) | (F.col("CANCELLED") == 1), 1)
        .otherwise(0)
    ).alias("delayed_flights")
)

# change the column name for intuitive display
airport_performance = airport_performance.withColumnRenamed("ORIGIN", "airport_code")

# calculate percentage delayed and average flights per day
airport_performance = airport_performance.withColumn(
    "pct_delayed",
    (F.col("delayed_flights") / F.col("total_flights")) * 100
).withColumn(
    "avg_flights_per_day",
    F.col("total_flights") / F.lit(total_days)
)

# select relevant columns and get top 50 by delay percentage
top_50_airports = airport_performance.select(
    "airport_code",
    "pct_delayed",
    "avg_flights_per_day",
    "total_flights"
).orderBy(F.desc("pct_delayed")).limit(50).show(50)

